# ddharmon v1 — Sub-cluster-anchored CDE Harmonization

**Canonical end-to-end pipeline.** Clusters cohort variables *and* NIH CDEs together using
**dual vectors** (separate semantic + value-encoding embeddings), **sub-clusters by value
vectors**, **recommends a CDE per sub-cluster**, and emits an **LLM adopt / refine / novel**
recommendation per sub-cluster — routed to **EITL** (expert-in-the-loop) for human verification.

```
ingest (cohorts + CDE)
  → dual-vector embed (semantic + value)
    → semantic cluster (BERTopic)
      → value sub-cluster (HDBSCAN on value vectors, per topic)
        → CDE anchor per sub-cluster (medoid → best in-cluster CDE; GenCDE fallback)
          → adopt / refine / novel  (single classify-only LLM call)
            → EITL review queue
```

**Lineage.** v1 is a deliberately-scoped extension of the embedding-clustering-for-variable-
harmonization line of work — **Krishnamurthy 2025** (arXiv:2506.02160; CDE-side clustering) and
**Salimi 2025** (*Sci Rep*; PD-cohort clustering with HDBSCAN). We cluster *cohort* variables
(source side), sub-cluster by value encoding, and anchor each sub-cluster to a CDE.

**In v1:** ingestion · dual-vector embedding · BERTopic · value sub-clustering · CDE anchoring ·
A/R/N classify → EITL.
**Out (publication-pending):** the LLM *coherence judge*, LLM concept-labeling, LLM
spec authoring, granularity-loss detection, deep recursive clustering, CDE CDM.

The reusable logic lives in `ddharmon.harmonization` and `ddharmon.clustering`; this notebook is
a thin orchestration over that API.

In [ ]:
!pip install --force-reinstall "ddharmon[clustering,bertopic] @ git+https://github.com/Phenome-Health/ddharmon.git"
!pip install sentence-transformers

In [ ]:
import json
import logging
from pathlib import Path

from ddharmon.ingestion import load_dictionary
from ddharmon.embedding import SentenceTransformerProvider, embed_dictionary
from ddharmon.harmonization import (
    harmonize_dictionaries,
    assemble_verdicts,
    write_prompts_jsonl,
    write_buckets,
    export_eitl_queue,
)
from ddharmon.llm import submit_and_wait  # Anthropic Batch API (needs ANTHROPIC_API_KEY)

try:
    from ddharmon.ingestion import preprocess_dictionary
except ImportError:
    preprocess_dictionary = None

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
print("Imports OK")

## 1. Load cohorts + CDE catalog

CDEs are loaded as a cohort named **`NIH_CDE`** so they participate in clustering — each
sub-cluster's anchor is the best CDE that lands *in that sub-cluster*. Choose the CDE set:

- `all_cdes_flat.tsv` — full repo (~22.7k CDEs), Krishnamurthy-scale, maximal anchor coverage.
- `nih_endorsed_flat.tsv` — the curated endorsed subset (~174), faster.

Generate the flat TSVs once with `python scripts/flatten_cde_repo.py <input.json> <output.tsv>`.


In [ ]:
DATA_DIR = Path("data/final combined responses")

USE_FULL_CDE = True  # True -> all_cdes_flat.tsv (~22.7k); False -> nih_endorsed_flat.tsv (~174)
CDE_FILE = DATA_DIR / ("all_cdes_flat.tsv" if USE_FULL_CDE else "nih_endorsed_flat.tsv")

loaders = {
    "NIH_CDE": (CDE_FILE, dict(
        variable_name="designation", field_id="tinyId",
        description="definition", question_text="question_text",
        data_type="datatype", value_encoding="permissible_values",
        category="classification", standard_code="concept_codes",
        embed_variable_name=True,
    )),
    "TwinsUK": (DATA_DIR / "twinsuk_selfreported.csv", dict(
        variable_name="Historical_ID", description="Phenotype_Description",
        data_type="Data_Type", standard_code="snomed_term_1",
    )),
    "Arivale": (DATA_DIR / "arivale_combined_responses.tsv", dict(
        variable_name="Column Name", description="Description",
        category="Category", data_type="Variable Type", units="Units",
        value_encoding="answer_options",
    )),
    "HPP": (DATA_DIR / "israeli10k_combined.tsv", dict(
        variable_name="field_name", short_label="field_string",
        description="field_description", category="category",
        data_type="data_type_pandas", units="units", coding_id="data_coding",
    )),
    "UKBB": (DATA_DIR / "ukbb_combined_responses.tsv", dict(
        variable_name="field_name", field_id="field_id",
        description="description", category="parent_category",
        data_type="data_type", units="units",
        value_encoding="response_options", coding_id="value_encoding",
    )),
    "AllOfUs": (DATA_DIR / "all_of_us_surveys.csv", dict(
        variable_name="Item Concept", description="Field Label",
        category="Survey", data_type="Field Type",
        value_encoding="Choices, Calculations, OR Slider Labels",
        question_text="Field Label",
    )),
}

cohorts = {}
for name, (path, kwargs) in loaders.items():
    if not path.exists():
        print(f"{name}: SKIPPED (not found: {path})")
        continue
    dd = load_dictionary(path, cohort_name=name, **kwargs)
    if preprocess_dictionary is not None:
        dd = preprocess_dictionary(dd)
    cohorts[name] = dd
    print(f"{name}: {dd.field_count} fields")

print(f"\nTotal: {sum(d.field_count for d in cohorts.values())} fields across {len(cohorts)} cohorts")

## 2. Embed — dual vectors (semantic + value)

Each field gets two vectors: a **semantic** vector (question/description meaning) and a **value**
vector (response-option / encoding structure). BERTopic clusters on the *semantic* vectors only;
the *value* vectors drive sub-clustering in step 4. Embeddings are SQLite-cached, so re-runs only
embed new/changed fields.


In [ ]:
provider = SentenceTransformerProvider()
embedded = []
for name, dd in cohorts.items():
    ed = embed_dictionary(dd, provider=provider)
    embedded.append(ed)
    print(f"  {name}: {len(ed.embeddings)} semantic, {len(ed.value_embeddings)} value vectors")
print(f"\n{len(embedded)} dictionaries embedded.")

## 3–6. Run the pipeline (cluster → sub-cluster → anchor → classify)

`harmonize_dictionaries` runs the whole chain. The single LLM call is the **classify-only**
adopt/refine/novel pass; we submit it via the Anthropic **Batch API** (≈50% cheaper, resumable).
Sub-clusters that can't be classified by an LLM are decided deterministically:

- **single_cohort** — only one cohort present (no cross-cohort pooling) → skipped.
- **novel / unaligned** — no candidate CDE in the sub-cluster (GenCDE needed) → forced verdict.
- **cde_only / noise** — no cohort data / HDBSCAN noise → flagged, not harmonized.

> **Air-gapped?** Call `harmonize_dictionaries(embedded, classify=None)` to get back
> `result.prompt_records`, export with `write_prompts_jsonl`, run
> `scripts/process_prompts_batch.sh` on a connected host, then `assemble_verdicts(...)`.


In [ ]:
WORK = Path("harmonization_artifacts")
WORK.mkdir(exist_ok=True)


def classify_via_batch(records):
    """Classify A/R/N prompts via the Anthropic Batch API. Returns {id: response}."""
    if not records:
        return {}
    prompts_path = WORK / "prompts_harmonize_arn.jsonl"
    responses_path = WORK / "responses_harmonize_arn.jsonl"
    write_prompts_jsonl(records, prompts_path)
    submit_and_wait(prompts_path, responses_path)  # needs ANTHROPIC_API_KEY
    responses = {}
    with open(responses_path) as f:
        for line in f:
            rec = json.loads(line)
            responses[rec["id"]] = rec["response"]
    return responses


result = harmonize_dictionaries(embedded, classify=classify_via_batch, min_cluster_size=15)
print(f"{len(result.verdicts)} sub-cluster verdicts ({len(result.prompt_records)} via LLM)")

## 7. Outputs → buckets + EITL review queue

`write_buckets` writes one JSON per verdict bucket; `export_eitl_queue` writes a TSV review queue
(refine/novel first) for the expert-in-the-loop app. **Nothing is auto-applied** — every
recommendation is human-verified downstream.


In [ ]:
counts = write_buckets(result, WORK / "buckets")
n_eitl = export_eitl_queue(result, WORK / "eitl_queue.tsv")
print("Buckets:", counts)
print(f"EITL queue: {n_eitl} sub-clusters -> {WORK / 'eitl_queue.tsv'}")

## Inspect verdicts

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "sub_cluster": v.sub_cluster_id,
        "label": v.label,
        "verdict": v.verdict or "(skipped)",
        "mode": v.mode,
        "parent_cde_id": v.parent_cde_id,
        "anchor": v.anchor_designation,
        "confidence": v.confidence,
        "n_fields": v.n_fields,
        "cohorts": ";".join(v.cohorts),
        "decided_by": v.decided_by,
    }
    for v in result.verdicts
])
print(df["verdict"].value_counts())
df.sort_values(["verdict", "confidence"], ascending=[True, False]).head(30)